# [1] OpenCV: 태권도 사진 전처리하고 사람 부분 잘라내기

데이터: AI Hub 태권도 동작 - **뒷굽이하고 바깥막기** (사진 1,197장 + json 라벨)

파일 이름 끝 글자가 동작 순간이에요.
- `S01` = 시작 자세 / `M01` = 중간 자세 / `E01` = 끝 자세

이 노트북에서 할 것
1. json 라벨 읽어서 박스 + 관절 점 그리기
2. 초록 배경(크로마키)을 HSV로 지워서 사람만 남기기
3. 사람 부분만 잘라서 `tk_crops/1_start, 2_middle, 3_end` 폴더에 저장 → [2], [3] 노트북에서 사용

In [4]:
import os
import glob
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

In [5]:
# 경로 설정 (각자 컴퓨터에 맞게 바꾸기)
DATA_ROOT = os.path.join("..", "taekwondo_data")   # 이 노트북 폴더 기준 한 칸 위 (각자 경로에 맞게)
IMG_ROOT = os.path.join(DATA_ROOT, "원천데이터")
LABEL_ROOT = os.path.join(DATA_ROOT, "라벨링데이터")

OUT_DIR = "tk_crops"     # 잘라낸 사진 저장 폴더 (노트북 옆에 생김)
IMG_SIZE = 128           # 저장할 크기 128 x 128

# 파일 이름 끝 글자 → 저장할 폴더 이름 (앞 숫자는 순서 맞추려고 붙임)
CLASS_DIR = {"S": "1_start", "M": "2_middle", "E": "3_end"}

In [6]:
# cv2.imread는 경로에 한글이 있으면 None을 돌려줌 → numpy로 읽어서 디코딩
def imread_kr(path):
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def imwrite_kr(path, img):
    ok, buf = cv2.imencode(".jpg", img)
    buf.tofile(path)

# OpenCV는 BGR, matplotlib은 RGB
def show(img, title=""):
    plt.figure(figsize=(8, 5))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1) json 라벨 읽기

In [7]:
# 라벨 폴더 안의 json을 하위 폴더까지 전부 찾기
json_list = sorted(glob.glob(os.path.join(LABEL_ROOT, "**", "*.json"), recursive=True))
print("json 개수:", len(json_list))

# json 경로 → 같은 이름의 jpg 경로 (TL2 → TS1, .json → .jpg)
def json_to_jpg(json_path):
    rel = os.path.relpath(json_path, LABEL_ROOT)          # TL2\뒷굽이...\A035 ...\파일.json
    rel = rel.replace("TL2", "TS1", 1).replace(".json", ".jpg")
    return os.path.join(IMG_ROOT, rel)

print(json_list[0])
print(json_to_jpg(json_list[0]))

json 개수: 1197
..\taekwondo_data\라벨링데이터\TL2\뒷굽이하고 바깥막기\A035 태극4장 11품\P-004-011-A035-M-B2006-S-20211109-01-01-E01.json
..\taekwondo_data\원천데이터\TS1\뒷굽이하고 바깥막기\A035 태극4장 11품\P-004-011-A035-M-B2006-S-20211109-01-01-E01.jpg


In [8]:
with open(json_list[0], encoding="utf-8") as f:
    label = json.load(f)

# labelingInfo 안에 pose(관절), box(박스), categories(품새) ... 가 리스트로 들어 있음
for item in label["labelingInfo"]:
    print(list(item.keys()))

['pose']
['box']
['categories']
['images']
['videos']
['actors']
['info']


In [9]:
def read_label(json_path):
    with open(json_path, encoding="utf-8") as f:
        label = json.load(f)
    box, points = None, {}
    for item in label["labelingInfo"]:
        if "box" in item:
            b = item["box"]["boundingBox"]
            box = (b["Xmin"], b["Ymin"], b["Xmax"], b["Ymax"])
        if "pose" in item:
            for name, p in item["pose"]["location"].items():
                points[name] = (int(p["x"]), int(p["y"]))
    return box, points

box, points = read_label(json_list[0])
print("박스:", box)
print("관절 점 개수:", len(points))
print("예) 코:", points["코"], " 왼쪽 손목:", points["왼쪽 손목"])

박스: (403, 287, 913, 915)
관절 점 개수: 29
예) 코: (598, 331)  왼쪽 손목: (511, 536)


## 2) 박스와 관절 점 그리기

In [10]:
# 관절을 이어서 뼈대처럼 그릴 쌍
SKELETON = [("왼쪽 어깨", "오른쪽 어깨"), ("왼쪽 어깨", "왼쪽 팔꿈치"), ("왼쪽 팔꿈치", "왼쪽 손목"),
            ("오른쪽 어깨", "오른쪽 팔꿈치"), ("오른쪽 팔꿈치", "오른쪽 손목"),
            ("왼쪽 어깨", "왼쪽 엉덩이"), ("오른쪽 어깨", "오른쪽 엉덩이"), ("왼쪽 엉덩이", "오른쪽 엉덩이"),
            ("왼쪽 엉덩이", "왼쪽 무릎"), ("왼쪽 무릎", "왼쪽 발목"),
            ("오른쪽 엉덩이", "오른쪽 무릎"), ("오른쪽 무릎", "오른쪽 발목")]

def draw_label(img, box, points):
    out = img.copy()                                   # 원본 보호
    x1, y1, x2, y2 = box
    cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 3)          # 빨간 박스
    for a, b in SKELETON:
        if a in points and b in points:
            cv2.line(out, points[a], points[b], (255, 255, 0), 3)   # 하늘색 뼈대
    for p in points.values():
        cv2.circle(out, p, 6, (0, 255, 255), -1)                  # 노란 점 (-1 = 속 채우기)
    return out

# 같은 동작의 시작/중간/끝 3장 비교
plt.figure(figsize=(15, 4))
for k, key in enumerate(["S01", "M01", "E01"]):
    jp = json_list[0][:-8] + key + ".json"          # 파일 이름 끝(S01·M01·E01)만 바꾸기
    img = imread_kr(json_to_jpg(jp))
    box, points = read_label(jp)
    x1, y1, x2, y2 = box
    drawn = draw_label(img, box, points)[y1 - 50:y2 + 50, x1 - 50:x2 + 50]   # 사람 근처만 확대
    plt.subplot(1, 3, k + 1)
    plt.imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
    plt.title(f"{key} ({CLASS_DIR[key[0]]})")
    plt.axis("off")
plt.show()

## 3) 초록 배경 지우기 (HSV + inRange)
배경이 초록색 크로마키라서, **초록색인 곳 = 배경**으로 골라내면 사람만 남아요.

In [11]:
img = imread_kr(json_to_jpg(json_list[0]))
box, points = read_label(json_list[0])

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# OpenCV의 H(색상)는 0~179. 초록은 대략 35~85
lower_green = np.array([35, 60, 40])
upper_green = np.array([85, 255, 255])
green_mask = cv2.inRange(hsv, lower_green, upper_green)    # 초록 = 255(흰색)

person_mask = cv2.bitwise_not(green_mask)                   # 반전 → 사람(+초록 아닌 것) = 255

# 모폴로지로 다듬기: 작은 점 지우고(open) 구멍 메우기(close)
kernel = np.ones((5, 5), np.uint8)
person_mask = cv2.morphologyEx(person_mask, cv2.MORPH_OPEN, kernel)
person_mask = cv2.morphologyEx(person_mask, cv2.MORPH_CLOSE, kernel)

person_only = cv2.bitwise_and(img, img, mask=person_mask)  # 마스크가 흰 곳만 남기기

plt.figure(figsize=(15, 4))
for k, (im, t) in enumerate([(img, "원본"), (green_mask, "초록 마스크"), (person_only, "배경 지운 결과")]):
    plt.subplot(1, 3, k + 1)
    plt.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap="gray")
    plt.title(t); plt.axis("off")
plt.show()

In [12]:
# 윤곽선으로 사람 위치 찾기 → 라벨 박스와 비교
# 검은 띠·머리카락 때문에 사람이 여러 조각으로 나뉠 수 있어서, 크게 close 해서 이어 붙임
mask2 = cv2.morphologyEx(person_mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
contours, _ = cv2.findContours(mask2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 화면 가장자리에 붙은 것(카메라 장비, 검은 테두리)과 작은 점은 빼고 나머지 조각을 사람으로 봄
H, W = img.shape[:2]
person_parts = []
for c in contours:
    x, y, w, h = cv2.boundingRect(c)
    touch_edge = x <= 5 or y <= 5 or x + w >= W - 5 or y + h >= H - 5
    if cv2.contourArea(c) > 1000 and not touch_edge:
        person_parts.append(c)
print("사람 조각 수:", len(person_parts))

px, py, pw, ph = cv2.boundingRect(np.vstack(person_parts))   # 조각들을 모두 감싸는 박스

x1, y1, x2, y2 = box
out = img.copy()
cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 3)                 # 빨강: 라벨 박스
cv2.rectangle(out, (px, py), (px + pw, py + ph), (255, 0, 0), 3)       # 파랑: OpenCV로 찾은 박스
cv2.drawContours(out, person_parts, -1, (0, 255, 255), 2)              # 노랑: 윤곽선

# IoU: 두 박스가 얼마나 겹치는지 (1이면 완전히 같음)
ix = max(0, min(x2, px + pw) - max(x1, px))
iy = max(0, min(y2, py + ph) - max(y1, py))
inter = ix * iy
iou = inter / ((x2 - x1) * (y2 - y1) + pw * ph - inter)
print(f"라벨 박스 {box} / OpenCV 박스 {(px, py, px + pw, py + ph)} / IoU = {iou:.2f}")
# 라벨 박스는 사람 둘레에 여유를 두고 크게 쳐져 있어서, 딱 맞는 OpenCV 박스와 IoU가 0.5 안팎으로 나옴

show(out[y1 - 80:y2 + 80, x1 - 80:x2 + 80], "빨강=라벨, 파랑=OpenCV, 노랑=윤곽선")

사람 조각 수: 2
라벨 박스 (403, 287, 913, 915) / OpenCV 박스 (437, 256, 839, 837) / IoU = 0.66


## 4) 사람 부분만 잘라서 저장
라벨 박스를 기준으로 **정사각형**으로 잘라요. (사람은 세로로 길어서 그냥 자르고 줄이면 찌그러짐)

In [13]:
def crop_square(img, box, margin=0.1):
    H, W = img.shape[:2]
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    half = int(max(x2 - x1, y2 - y1) * (1 + margin) / 2)    # 긴 변 기준 + 여유 10%
    # 사진 밖으로 나가지 않게 0 ~ W, 0 ~ H 사이로
    left, right = max(0, cx - half), min(W, cx + half)
    top, bottom = max(0, cy - half), min(H, cy + half)
    return img[top:bottom, left:right]

crop = crop_square(img, box)
crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
print("잘라낸 크기:", crop.shape)
show(crop, "128 x 128")

잘라낸 크기: (128, 128, 3)


In [14]:
# 전체 저장 (1,197장이라 1~2분 걸림)
for d in CLASS_DIR.values():
    os.makedirs(os.path.join(OUT_DIR, d), exist_ok=True)

count = {d: 0 for d in CLASS_DIR.values()}
for jp in json_list:
    key = os.path.basename(jp)[-8]            # ...-S01.json → 'S'
    if key not in CLASS_DIR:
        continue
    ip = json_to_jpg(jp)
    if not os.path.exists(ip):
        continue
    img = imread_kr(ip)
    box, _ = read_label(jp)
    crop = cv2.resize(crop_square(img, box), (IMG_SIZE, IMG_SIZE))
    name = os.path.basename(ip)
    imwrite_kr(os.path.join(OUT_DIR, CLASS_DIR[key], name), crop)
    count[CLASS_DIR[key]] += 1

print(count)

{'1_start': 399, '2_middle': 399, '3_end': 399}


In [15]:
# 클래스별로 5장씩 확인
plt.figure(figsize=(12, 7))
for r, d in enumerate(CLASS_DIR.values()):
    files = sorted(os.listdir(os.path.join(OUT_DIR, d)))[:5]
    for c, f in enumerate(files):
        im = imread_kr(os.path.join(OUT_DIR, d, f))
        plt.subplot(3, 5, r * 5 + c + 1)
        plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        plt.title(d, fontsize=9); plt.axis("off")
plt.tight_layout(); plt.show()